In [1]:
# convert sales agent as tools
# write a function for sending email
# add all these in a tools list
# create an agent for sales manager and ask to send the best email using tools
# check the trace

In [2]:
import os
import asyncio
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
from dotenv import load_dotenv

In [3]:
from agents import Agent, Runner, trace, function_tool

In [3]:
# from openai.types.responses import ResponseTextDeltaEvent

In [4]:
load_dotenv(override=True)

True

In [5]:
# def send_test_email():
#     sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
#     from_email = Email("wetechfin@gmail.com")
#     to_email = To("debmalyamondal63@gmail.com")
#     content = Content("text/plain", "This is a the test email body")
#     mail = Mail(from_email, to_email, "test email", content).get()
#     response = sg.client.mail.send.post(request_body=mail)
#     print(response.status_code)

In [6]:
# send_test_email()

In [7]:
instruction1 = "You are a sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium business. \
You write professional, serious cold emails"

instruction2 = "You are a humorous, engaging sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium businesses. \
You write witty, engaging cold emails that are likely to get a response."

instruction3 = "You are a busy sales agent working for AutAI, \
an AI automation agency company that provides automated solution for small and medium businesses. \
You write concise, to the point cold emails"



In [8]:
sales_agent1 = Agent(
    name="Professional sales agent",
    instructions=instruction1,
    model="gpt-4o-mini"
)

sales_agent2 = Agent(
    name="Engaging sales agent",
    instructions=instruction2,
    model="gpt-4o-mini"
)

sales_agent3 = Agent(
    name="Busy sales agent",
    instructions=instruction3,
    model="gpt-4o-mini"
)


In [9]:
description="write a cold email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

In [19]:
@function_tool
def send_email(body: str):
    """Send out email"""
    sg = sendgrid.SendGridAPIClient(api_key=os.getenv("SENDGRID_API_KEY"))
    from_email = Email("wetechfin@gmail.com")
    to_email = To("debmalyamondal63@gmail.com")
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [20]:
tools = [tool1, tool2, tool3, send_email]

In [21]:
instructions = """
You are a sales manager at the Automation Agency company AutAI. Your goal is to find the single best
sales email using the sales_agent tools.
Follow these instructions carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different cold email drafts. 
Do not proceed until all three drafts are ready.
2. Evaluate and Select: Review and select the best email using your judgement of which one is more effective.
3. Use the send_email tool to send the best email and only the best email.

Crucial Rules:
- You must use the sales_agent tools to generate the drafts. Do not write them yourself.
- You must send one email using send_email tool and not more than one.
"""

In [22]:
sales_manager = Agent(
    name="sales_manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini"
)

In [23]:
message = "Send a cold email addressed to 'Dear CEO'"

with trace("AutAI Sales Manager"):
    result = await Runner.run(sales_manager, message)

In [24]:
result.final_output

'The selected email has been successfully sent to the CEO. If you need further assistance or additional tasks, feel free to ask!'